In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

# Data

In [3]:

# 2. DEFINE KEY PARAMETERS
# --------------------------
# File path for the dataset
# Note: Python uses forward slashes '/' for paths, even on Windows.
FILE_PATH = "../../Data/platinum/dengue_weather.csv"

# Proportion of the time period to use for the final test set
TEST_PROPORTION = 0.20

# Set a seed for all random processes for reproducibility
# This is equivalent to R's set.seed()
np.random.seed(42)


# 3. DATA LOADING AND PRE-PROCESSING
# ------------------------------------
print("--- Loading and pre-processing data ---")

# Load the raw data using pandas
full_df = pd.read_csv(FILE_PATH)

--- Loading and pre-processing data ---


# Processing

In [4]:

# Convert the 'week' column from object/string to datetime
# The format string helps pandas parse the date correctly.
full_df['week'] = pd.to_datetime(full_df['week'], format="%m/%d/%Y")

# Create a numeric representation of the date (e.g., POSIX timestamp)
# This is useful for modeling time as a continuous variable.
full_df['week_num'] = full_df['week'].apply(lambda x: x.timestamp())

# Create 'departamento' column
full_df['departamento'] = full_df['idx_city'].str.split('_').str[1]

# Convert the pandas DataFrame to a GeoDataFrame
# The coordinate system (CRS) "EPSG:4326" is standard for Latitude/Longitude data.
geometry = [Point(xy) for xy in zip(full_df['Longitude'], full_df['Latitude'])]
full_gdf_pre = gpd.GeoDataFrame(full_df, geometry=geometry, crs="EPSG:4326")


# Create the t+1 target variable ('count_t1')
# We sort by city and time to ensure the shift is applied correctly within each group.
full_gdf_pre = full_gdf_pre.sort_values(by=['idx_city', 'week_num'])
full_gdf_pre['count_t1'] = full_gdf_pre.groupby('idx_city')['count'].shift(-1)

# Remove the last observation for each city, which now has a NaN for 'count_t1'
full_gdf = full_gdf_pre.dropna(subset=['count_t1']).copy()


print("Data loaded, converted to spatial object, and t+1 target created.")

Data loaded, converted to spatial object, and t+1 target created.


## Train test

In [5]:
# 4. TRAIN-TEST SPLIT BY TIME
# -----------------------------
# This creates the final hold-out test set based on a time cutoff.

print("\n--- Creating train-test split based on time ---")

# Find the cutoff week number that separates the data
time_range_min = full_gdf['week_num'].min()
time_range_max = full_gdf['week_num'].max()

# Calculate the cutoff timestamp
cutoff_week = time_range_min + (1 - TEST_PROPORTION) * (time_range_max - time_range_min)

# Create the training and testing sets based on the time cutoff
train_data = full_gdf[full_gdf['week_num'] <= cutoff_week]
test_data = full_gdf[full_gdf['week_num'] > cutoff_week]

# Report summary of the split
print("Split complete:")
print(f"  - Training data: {len(train_data)} rows up to week number {round(cutoff_week)}")
print(f"  - Testing data:  {len(test_data)} rows after week number {round(cutoff_week)}")


--- Creating train-test split based on time ---
Split complete:
  - Training data: 10701 rows up to week number 1696809600
  - Testing data:  2610 rows after week number 1696809600


## Clean nans

In [7]:
# 5. CLEAN NaNs FROM DATA
# ------------------------------
predictor_cols = [
    'count', 'Latitude', 'Longitude', 'week_num', 'tavg', 'tmax',
    'prcp', 'wdir', 'wspd', 'pres', 'elevation'
]
# Also clean the training data
train_data_clean = train_data.dropna(subset=predictor_cols)
test_data_clean = test_data.dropna(subset=predictor_cols)

print(f"\nCleaned training data has {len(train_data_clean)} rows after removing NaNs.")
print(f"Cleaned test data has {len(test_data_clean)} rows after removing NaNs.")


Cleaned training data has 10681 rows after removing NaNs.
Cleaned test data has 2570 rows after removing NaNs.


# No count

## Define vars

In [8]:
# 6. DEFINE PREDICTORS AND TARGET
# ---------------------------------
# Based on the R formula:
# count_t1 ~ te(Longitude, Latitude) + s(tmax) + s(prcp) + s(wspd) + s(week_num)

# List of predictor variable names
features = ['Longitude', 'Latitude', 'tmax', 'prcp', 'wspd', 'week_num']

# Define the target variable name
target = 'count_t1'

# Create the training sets
X_train = train_data_clean[features]
y_train = train_data_clean[target]

# Create the testing sets
X_test = test_data_clean[features]
y_test = test_data_clean[target]

In [9]:
X_train

,Longitude,Latitude,tmax,prcp,wspd,week_num
0,-69.942596,-4.212921,29.171429,8.528571,4.800000,1.672618e+09
1,-69.942596,-4.212921,29.042857,12.128571,4.228571,1.673222e+09
2,-69.942596,-4.212921,28.714286,12.971429,3.800000,1.673827e+09
3,-69.942596,-4.212921,30.471429,12.671429,4.200000,1.674432e+09
4,-69.942596,-4.212921,29.228571,16.728571,4.028571,1.675037e+09
...,...,...,...,...,...,...
9240,-67.484189,6.190923,33.800000,4.971429,6.342857,1.694390e+09
9241,-67.484189,6.190923,31.957143,3.942857,6.100000,1.694995e+09
9242,-67.484189,6.190923,35.014286,5.742857,7.728571,1.695600e+09
9243,-67.484189,6.190923,35.385714,1.700000,7.528571,1.696205e+09


## Models

### OLS

In [10]:
# ols model with sklearn
from sklearn.linear_model    import LinearRegression
from sklearn.metrics         import mean_squared_error, r2_score

In [12]:
ols = LinearRegression()
ols.fit(X_train, y_train)
# get metrics on test set
y_pred = ols.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"R^2: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")

R^2: 0.002
RMSE: 32.448


### Lasso

In [13]:
from sklearn.pipeline      import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model  import LassoCV

In [14]:
pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('lasso', LassoCV())
])

pipeline.fit(X_train, y_train)
best_alpha = pipeline.named_steps["lasso"].alpha_
# get metrics on test set
y_pred = pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"R^2: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")

R^2: -0.007
RMSE: 32.590


### Ridge

In [15]:
from sklearn.linear_model  import RidgeCV

In [16]:
pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('ridge', RidgeCV())
])

pipeline.fit(X_train, y_train)
best_alpha = pipeline.named_steps["ridge"].alpha_

# get metrics on test set
y_pred = pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"R^2: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")


R^2: 0.002
RMSE: 32.448


### Elastic Net

In [17]:

from sklearn.linear_model  import ElasticNetCV

In [18]:
pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('elastic', ElasticNetCV())
])

pipeline.fit(X_train, y_train)
best_alpha = pipeline.named_steps["elastic"].alpha_

# get metrics on test set
y_pred = pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"R^2: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")


R^2: -0.008
RMSE: 32.600


### Random Forest

In [20]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble        import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit

In [21]:
param_grid = {
    "n_estimators":      [400, 800, 1200],
    "max_depth":         [None, 10, 20],
    "max_features":      ["sqrt", 0.6],
    "min_samples_leaf":  [1, 3, 5],
    "bootstrap":         [True, False]
}

rf_base  = RandomForestRegressor(random_state=42, n_jobs=-1)
inner_cv = TimeSeriesSplit(n_splits=3)

grid = GridSearchCV(
    estimator   = rf_base,
    param_grid  = param_grid,
    cv          = inner_cv,
    scoring     = "neg_root_mean_squared_error",
    n_jobs      = -1,
    verbose     = 0
)

grid.fit(X_train, y_train)
best_rf     = grid.best_estimator_
best_params = grid.best_params_

# get metrics
y_pred = best_rf.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"R^2: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")


R^2: 0.499
RMSE: 22.997


### Gradient Boosting

In [23]:
from sklearn.ensemble        import GradientBoostingRegressor

In [24]:
param_grid = {
    "n_estimators":     [200, 400, 800],
    "learning_rate":    [0.03, 0.05, 0.1],
    "max_depth":        [2, 3, 4],
    "subsample":        [0.7, 0.9, 1.0],
    "min_samples_leaf": [1, 3, 5],
    "max_features":     ["sqrt", 0.6, None]
}

inner_cv = TimeSeriesSplit(n_splits=3)
gb_base  = GradientBoostingRegressor(random_state=42)

grid = GridSearchCV(
    estimator   = gb_base,
    param_grid  = param_grid,
    cv          = inner_cv,
    scoring     = "neg_root_mean_squared_error",
    n_jobs      = -1,
    verbose     = 0
)
grid.fit(X_train, y_train)
best_gb     = grid.best_estimator_
best_params = grid.best_params_

# get metrics
y_pred = best_gb.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"R^2: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")


R^2: 0.146
RMSE: 30.008


### SVM

In [25]:
from sklearn.svm            import SVR

In [26]:
param_grid = [
    {   # RBF search
        "model__kernel":  ["rbf"],
        "model__C":       [0.5, 1, 5, 20],
        "model__epsilon": [0.05, 0.1, 0.2],
        "model__gamma":   ["scale", 0.01, 0.1, 1.0],
    },
    {   # Linear search (no gamma)
        "model__kernel":  ["linear"],
        "model__C":       [0.5, 1, 5, 20],
        "model__epsilon": [0.05, 0.1, 0.2],
    },
]

inner_cv = TimeSeriesSplit(n_splits=3)

svr_pipe = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model",  SVR())
    ]
)

grid = GridSearchCV(
    estimator   = svr_pipe,
    param_grid  = param_grid,
    cv          = inner_cv,
    scoring     = "neg_root_mean_squared_error",
    n_jobs      = -1,
    verbose     = 0,
)
grid.fit(X_train, y_train)

best_svr    = grid.best_estimator_
best_params = grid.best_params_

# get metrics
y_pred = best_svr.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"R^2: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")


R^2: -0.015
RMSE: 32.720


# Count

In [28]:
# 6. DEFINE PREDICTORS AND TARGET
# ---------------------------------
# Based on the R formula:
# count_t1 ~ te(Longitude, Latitude) + s(tmax) + s(prcp) + s(wspd) + s(week_num)

# List of predictor variable names
features = ['Longitude', 'Latitude', 'tmax', 'prcp', 'wspd', 'week_num', 'count']

# Define the target variable name
target = 'count_t1'

# Create the training sets
X_train = train_data_clean[features]
y_train = train_data_clean[target]

# Create the testing sets
X_test = test_data_clean[features]
y_test = test_data_clean[target]

## Models

### OLS

In [29]:
# ols model with sklearn
from sklearn.linear_model    import LinearRegression
from sklearn.metrics         import mean_squared_error, r2_score

In [30]:
ols = LinearRegression()
ols.fit(X_train, y_train)
# get metrics on test set
y_pred = ols.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"R^2: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")

R^2: 0.973
RMSE: 5.356


### Lasso

In [31]:
from sklearn.pipeline      import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model  import LassoCV

In [32]:
pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('lasso', LassoCV())
])

pipeline.fit(X_train, y_train)
best_alpha = pipeline.named_steps["lasso"].alpha_
# get metrics on test set
y_pred = pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"R^2: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")

R^2: 0.973
RMSE: 5.362


### Ridge

In [33]:
from sklearn.linear_model  import RidgeCV

In [34]:
pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('ridge', RidgeCV())
])

pipeline.fit(X_train, y_train)
best_alpha = pipeline.named_steps["ridge"].alpha_

# get metrics on test set
y_pred = pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"R^2: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")


R^2: 0.973
RMSE: 5.362


### Elastic Net

In [35]:

from sklearn.linear_model  import ElasticNetCV

In [36]:
pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('elastic', ElasticNetCV())
])

pipeline.fit(X_train, y_train)
best_alpha = pipeline.named_steps["elastic"].alpha_

# get metrics on test set
y_pred = pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"R^2: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")


R^2: 0.971
RMSE: 5.493


### Random Forest

In [37]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble        import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit

In [38]:
param_grid = {
    "n_estimators":      [400, 800, 1200],
    "max_depth":         [None, 10, 20],
    "max_features":      ["sqrt", 0.6],
    "min_samples_leaf":  [1, 3, 5],
    "bootstrap":         [True, False]
}

rf_base  = RandomForestRegressor(random_state=42, n_jobs=-1)
inner_cv = TimeSeriesSplit(n_splits=3)

grid = GridSearchCV(
    estimator   = rf_base,
    param_grid  = param_grid,
    cv          = inner_cv,
    scoring     = "neg_root_mean_squared_error",
    n_jobs      = -1,
    verbose     = 0
)

grid.fit(X_train, y_train)
best_rf     = grid.best_estimator_
best_params = grid.best_params_

# get metrics
y_pred = best_rf.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"R^2: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")


R^2: 0.955
RMSE: 6.873


### Gradient Boosting

In [39]:
from sklearn.ensemble        import GradientBoostingRegressor

In [40]:
param_grid = {
    "n_estimators":     [200, 400, 800],
    "learning_rate":    [0.03, 0.05, 0.1],
    "max_depth":        [2, 3, 4],
    "subsample":        [0.7, 0.9, 1.0],
    "min_samples_leaf": [1, 3, 5],
    "max_features":     ["sqrt", 0.6, None]
}

inner_cv = TimeSeriesSplit(n_splits=3)
gb_base  = GradientBoostingRegressor(random_state=42)

grid = GridSearchCV(
    estimator   = gb_base,
    param_grid  = param_grid,
    cv          = inner_cv,
    scoring     = "neg_root_mean_squared_error",
    n_jobs      = -1,
    verbose     = 0
)
grid.fit(X_train, y_train)
best_gb     = grid.best_estimator_
best_params = grid.best_params_

# get metrics
y_pred = best_gb.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"R^2: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")


R^2: 0.966
RMSE: 6.005


### SVM

In [41]:
from sklearn.svm            import SVR

In [42]:
param_grid = [
    {   # RBF search
        "model__kernel":  ["rbf"],
        "model__C":       [0.5, 1, 5, 20],
        "model__epsilon": [0.05, 0.1, 0.2],
        "model__gamma":   ["scale", 0.01, 0.1, 1.0],
    },
    {   # Linear search (no gamma)
        "model__kernel":  ["linear"],
        "model__C":       [0.5, 1, 5, 20],
        "model__epsilon": [0.05, 0.1, 0.2],
    },
]

inner_cv = TimeSeriesSplit(n_splits=3)

svr_pipe = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model",  SVR())
    ]
)

grid = GridSearchCV(
    estimator   = svr_pipe,
    param_grid  = param_grid,
    cv          = inner_cv,
    scoring     = "neg_root_mean_squared_error",
    n_jobs      = -1,
    verbose     = 0,
)
grid.fit(X_train, y_train)

best_svr    = grid.best_estimator_
best_params = grid.best_params_

# get metrics
y_pred = best_svr.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"R^2: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")


R^2: 0.970
RMSE: 5.620
